In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn # For potential future machine learning tasks
import folium # For geographical visualizations
import plotly.express as px # For interactive plotting, if needed

# Configure plot styles for better readability
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
csv_file_path = '/content/Electric_Vehicle_Population_Data.csv'
try:
    df = pd.read_csv(csv_file_path)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please ensure it's uploaded.")
    df = pd.DataFrame() # Create an empty DataFrame to avoid errors


# Display the first few rows of the DataFrame
print("\n--- First 5 rows of the dataset ---")
display(df.head())

# Display the shape of the DataFrame (number of rows, number of columns)
print("\n--- Dataset Shape ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Display the column names
print("\n--- Column Names ---")
print(df.columns.tolist())

# Display concise summary of the DataFrame, including data types and non-null values
print("\n--- Dataset Information ---")
df.info()

# Display descriptive statistics for numerical columns
print("\n--- Descriptive Statistics ---")
display(df.describe(include='all'))

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("--- Columns with Missing Values ---")
display(missing_values)

print("\n--- Percentage of Missing Values ---")
display((missing_values / len(df) * 100).round(2))

# For simplicity, let's fill numerical missing values with the mean/median and drop rows with critical categorical missing values if their count is low.
# For 'Electric Range', let's fill with the median as 0 is a significant value.
df['Electric Range'].fillna(df['Electric Range'].median(), inplace=True)

# For 'Legislative District', 'Postal Code', '2020 Census Tract', fill with mode or consider more advanced imputation if needed. For now, fill with mode.
for col in ['Legislative District', 'Postal Code', '2020 Census Tract']:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

# For 'County', 'City', 'Vehicle Location', 'Electric Utility', which are categorical, if the number of missing values is small, we can drop them or fill with 'Unknown'.
# Given the small percentage (less than 0.01% for most), dropping them might be acceptable for these columns.
df.dropna(subset=['County', 'City', 'Vehicle Location', 'Electric Utility'], inplace=True)

print("\n--- Missing values after handling ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print(f"\nDataFrame shape after handling missing values: {df.shape}")

In [ ]:
# Clean column names
def clean_col_names(df):
    cols = df.columns
    new_cols = []
    for col in cols:
        new_col = col.lower().replace(' ', '_').replace('(1-10)', '').replace('/', '_').replace('(', '').replace(')', '').strip()
        new_cols.append(new_col)
    df.columns = new_cols
    return df

df = clean_col_names(df)

print("--- Cleaned Column Names ---")
print(df.columns.tolist())

In [ ]:
print(f"Initial number of rows: {df.shape[0]}")

df.drop_duplicates(inplace=True)

print(f"Number of rows after removing duplicates: {df.shape[0]}")

print("\n--- Dataset after cleaning and preprocessing (first 5 rows) ---")
display(df.head())
print("\n--- Data types after cleaning and preprocessing ---")
df.info()

In [ ]:
# Analyze 'electric_range' zero values
zero_range_vehicles = df[df['electric_range'] == 0]
print(f"Number of vehicles with 0 electric range: {len(zero_range_vehicles)}")
print("\n--- Electric Vehicle Type distribution for vehicles with 0 electric range ---")
display(zero_range_vehicles['electric_vehicle_type'].value_counts())

# Visualize the distribution of electric range
plt.figure(figsize=(10, 6))
sns.histplot(df['electric_range'], bins=50, kde=True)
plt.title('Distribution of Electric Range')
plt.xlabel('Electric Range (miles)')
plt.ylabel('Number of Vehicles')
plt.show()

In [ ]:
import hashlib

# Check uniqueness of original VIN
print(f"Number of unique VINs: {df['vin_'].nunique()} out of {len(df)} total records.")

# Create an anonymized VIN using a hash function
df['hashed_vin'] = df['vin_'].apply(lambda x: hashlib.sha256(str(x).encode()).hexdigest())

# Verify uniqueness of hashed VIN
print(f"Number of unique hashed VINs: {df['hashed_vin'].nunique()} out of {len(df)} total records.")

# Drop the original 'vin_' column as it's been anonymized
df.drop('vin_', axis=1, inplace=True)
print("Original 'vin_' column dropped.")

In [ ]:
# Extract Latitude and Longitude from 'vehicle_location'
def extract_lat_lon(location_str):
    if isinstance(location_str, str) and 'POINT (' in location_str:
        coords = location_str.replace('POINT (', '').replace(')', '').split()
        return float(coords[1]), float(coords[0])
    return np.nan, np.nan

df[['latitude', 'longitude']] = df['vehicle_location'].apply(lambda x: pd.Series(extract_lat_lon(x)))

print("--- First 5 rows with new 'latitude' and 'longitude' columns ---")
display(df[['vehicle_location', 'latitude', 'longitude']].head())

# Check for any remaining missing values in new columns after extraction
print("\n--- Missing values in 'latitude' and 'longitude' ---")
print(df[['latitude', 'longitude']].isnull().sum())

In [ ]:
output_csv_path = '/content/electric_vehicle_data_cleaned.csv'
df.to_csv(output_csv_path, index=False)
print(f"Cleaned dataset saved to {output_csv_path}")

## 5. Exploratory Data Analysis (EDA)



### 5.1 EV Registrations by Make



In [ ]:
# Count EV registrations by Make
ev_registrations_by_make = df['make'].value_counts()

print("--- EV Registrations by Make ---")
display(ev_registrations_by_make.head(10))

print("\n--- Top 5 EV Makes ---")
display(ev_registrations_by_make.head(5))

# Visualize top makes
plt.figure(figsize=(12, 7))
sns.barplot(x=ev_registrations_by_make.head(10).index, y=ev_registrations_by_make.head(10).values, palette='viridis')
plt.title('Top 10 EV Makes by Registration')
plt.xlabel('Make')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 5.2 EV Registrations by Model



In [ ]:
# Count EV registrations by Model
ev_registrations_by_model = df['model'].value_counts()

print("--- EV Registrations by Model ---")
display(ev_registrations_by_model.head(10))

print("\n--- Top 5 EV Models ---")
display(ev_registrations_by_model.head(5))

# Visualize top models
plt.figure(figsize=(12, 7))
sns.barplot(x=ev_registrations_by_model.head(10).index, y=ev_registrations_by_model.head(10).values, palette='magma')
plt.title('Top 10 EV Models by Registration')
plt.xlabel('Model')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 5.3 EV Registrations by County



In [ ]:
# Count EV registrations by County
ev_registrations_by_county = df['county'].value_counts()

print("--- EV Registrations by County ---")
display(ev_registrations_by_county.head(10))

print("\n--- County with Highest EV Count ---")
highest_county = ev_registrations_by_county.index[0]
highest_count = ev_registrations_by_county.values[0]
print(f"The county with the highest EV count is '{highest_county}' with {highest_count} registrations.")

# Visualize top counties
plt.figure(figsize=(12, 7))
sns.barplot(x=ev_registrations_by_county.head(10).index, y=ev_registrations_by_county.head(10).values, palette='cubehelix')
plt.title('Top 10 Counties by EV Registrations')
plt.xlabel('County')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 5.4 EV Adoption Trend Across Years



In [ ]:
# Group EVs by Model Year and study adoption trend
ev_adoption_by_year = df['model_year'].value_counts().sort_index()

print("--- EV Adoption by Model Year ---")
display(ev_adoption_by_year)

# Visualize adoption trend
plt.figure(figsize=(15, 7))
sns.lineplot(x=ev_adoption_by_year.index, y=ev_adoption_by_year.values, marker='o')
plt.title('EV Adoption Trend by Model Year')
plt.xlabel('Model Year')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 5.5 Average Electric Range



In [ ]:
# Calculate average Electric Range
average_electric_range = df['electric_range'].mean()
print(f"Average Electric Range: {average_electric_range:.2f} miles")

### 5.6 Analyze CAFV Eligibility



In [ ]:
# Analyze CAFV eligibility counts
cafv_eligibility_counts = df['clean_alternative_fuel_vehicle_cafv_eligibility'].value_counts()
print("--- CAFV Eligibility Counts ---")
display(cafv_eligibility_counts)

# Calculate CAFV eligibility percentage
cafv_eligibility_percentage = (cafv_eligibility_counts / len(df)) * 100
print("\n--- CAFV Eligibility Percentage ---")
display(cafv_eligibility_percentage.round(2))

# Visualize CAFV eligibility
plt.figure(figsize=(10, 6))
sns.barplot(x=cafv_eligibility_counts.index, y=cafv_eligibility_counts.values, palette='plasma')
plt.title('CAFV Eligibility Distribution')
plt.xlabel('Eligibility Status')
plt.ylabel('Number of Vehicles')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 5.7 Compare Electric Range by Make and Model



In [ ]:
# Compare Electric Range by Make
average_range_by_make = df.groupby('make')['electric_range'].mean().sort_values(ascending=False)
print("--- Average Electric Range by Make (Top 10) ---")
display(average_range_by_make.head(10))

# Visualize average electric range by make
plt.figure(figsize=(12, 7))
sns.barplot(x=average_range_by_make.head(10).index, y=average_range_by_make.head(10).values, palette='coolwarm')
plt.title('Average Electric Range by Make (Top 10)')
plt.xlabel('Make')
plt.ylabel('Average Electric Range (miles)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Compare Electric Range by Model
average_range_by_model = df.groupby('model')['electric_range'].mean().sort_values(ascending=False)
print("\n--- Average Electric Range by Model (Top 10) ---")
display(average_range_by_model.head(10))

# Visualize average electric range by model
plt.figure(figsize=(12, 7))
sns.barplot(x=average_range_by_model.head(10).index, y=average_range_by_model.head(10).values, palette='crest')
plt.title('Average Electric Range by Model (Top 10)')
plt.xlabel('Model')
plt.ylabel('Average Electric Range (miles)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 5.8 Analyze County-wise Adoption Patterns



In [ ]:
# For now, let's just re-display the top counties and perhaps consider a more granular look if an urban/rural indicator becomes available.
print("--- Top 10 Counties by EV Registrations (reiterating for context) ---")
display(df['county'].value_counts().head(10))

# A simple proxy for urban vs. rural could be the density of EV registrations or the presence of major cities.
# Let's count registrations by city for a more granular view.
ev_registrations_by_city = df['city'].value_counts()
print("\n--- Top 10 Cities by EV Registrations ---")
display(ev_registrations_by_city.head(10))

# To analyze urban vs. rural patterns, we would typically need external data on county/city classification or population density.
# Without such data, a direct comparison is challenging. However, we can infer that counties like 'King' (Seattle area) represent urban patterns.

# Let's examine the distribution of 'Electric Vehicle Type' across the top N counties as a proxy.
# This might reveal if BEVs or PHEVs are more prevalent in certain areas.

top_10_counties = df['county'].value_counts().head(10).index.tolist()
df_top_counties = df[df['county'].isin(top_10_counties)]

plt.figure(figsize=(14, 8))
sns.countplot(data=df_top_counties, x='county', hue='electric_vehicle_type', palette='viridis', order=top_10_counties)
plt.title('EV Type Distribution Across Top 10 Counties')
plt.xlabel('County')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45, ha='right')
plt.legend(title='EV Type')
plt.tight_layout()
plt.show()

## 6. Data Visualization



### 6.1 Top 5 EV Makes



In [ ]:
# Using the already computed ev_registrations_by_make
top_5_makes = ev_registrations_by_make.head(5)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_5_makes.index, y=top_5_makes.values, palette='viridis')
plt.title('Top 5 EV Makes by Registration')
plt.xlabel('Make')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.2 Top 5 EV Models



In [ ]:
# Using the already computed ev_registrations_by_model
top_5_models = ev_registrations_by_model.head(5)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_5_models.index, y=top_5_models.values, palette='magma')
plt.title('Top 5 EV Models by Registration')
plt.xlabel('Model')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.3 EV Registrations Distribution by County



In [ ]:
# Filter out rows with NaN latitude/longitude if any, though we handled them earlier.
df_geo = df.dropna(subset=['latitude', 'longitude'])

# Group by location for better visualization performance if there are many overlapping points
# Or directly plot if individual points are desired.
# For now, let's plot individual points, they will cluster by location naturally.

fig = px.scatter_mapbox(df_geo,
                        lat="latitude",
                        lon="longitude",
                        color="electric_vehicle_type", # Color by EV type
                        hover_name="county",
                        hover_data=["city", "make", "model"],
                        zoom=6,
                        height=600)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.update_layout(title_text='EV Registrations Geographical Distribution')
fig.show()

### 6.4 Yearly EV Adoption Trend



In [ ]:
# Using the already computed ev_adoption_by_year
plt.figure(figsize=(15, 7))
sns.lineplot(x=ev_adoption_by_year.index, y=ev_adoption_by_year.values, marker='o')
plt.title('EV Adoption Trend by Model Year')
plt.xlabel('Model Year')
plt.ylabel('Number of Registrations')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 6.5 CAFV Eligibility Pie Chart



In [ ]:
# Using the already computed cafv_eligibility_counts
plt.figure(figsize=(10, 8))
plt.pie(cafv_eligibility_counts.values, labels=cafv_eligibility_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('CAFV Eligibility Distribution')
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.tight_layout()
plt.show()

### 6.6 Geospatial Map using Folium



In [ ]:
import folium
from folium.plugins import MarkerCluster

# Create a base map centered around Washington State
m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], zoom_start=7)

# Create a MarkerCluster object
marker_cluster = MarkerCluster().add_to(m)

# Add markers to the cluster
for idx, row in df_geo.iterrows():
    if not pd.isna(row['latitude']) and not pd.isna(row['longitude']):
        popup_text = f"Make: {row['make']}<br>Model: {row['model']}<br>County: {row['county']}<br>City: {row['city']}"
        folium.Marker([row['latitude'], row['longitude']], popup=popup_text).add_to(marker_cluster)

# Display the map
m

## 7. Feature Engineering for Regression



### 7.1 Define Prediction Goal and Select Target Variable



In [ ]:
# Define the target variable
y = df['electric_range']

print(f"Target variable 'y' created with shape: {y.shape}")

### 7.2 Select Predictor Variables and Remove Unnecessary Columns



In [ ]:
# Select predictor variables
features = ['model_year', 'make', 'model', 'electric_vehicle_type']
X = df[features]

print(f"Selected features for 'X' with shape: {X.shape}")
print("Features:", X.columns.tolist())
display(X.head())

### 7.3 Handle Categorical Variables (One-Hot Encoding)



In [ ]:
# Apply one-hot encoding to categorical features
X = pd.get_dummies(X, columns=['make', 'model', 'electric_vehicle_type'], drop_first=True)

print(f"Feature matrix 'X' after one-hot encoding with shape: {X.shape}")
display(X.head())

### 7.4 Check Feature Matrix Shape



In [ ]:
print(f"Shape of feature matrix X: {X.shape}")
print(f"Shape of target variable y: {y.shape}")

## 8. Build Linear Regression Model



### 8.1 Train-Test Split



In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

### 8.2 Create and Train Linear Regression Model



In [ ]:
from sklearn.linear_model import LinearRegression

# Create a Linear Regression model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

### 8.3 Predict Electric Range



In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

print(f"Predictions made for {len(y_pred)} samples.")
print("First 5 predictions:", y_pred[:5])

### 8.4 Compare Actual vs. Predicted Values



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2) Score: {r2:.2f}")

# Visualize actual vs. predicted values (sample a subset if the test set is too large)
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Ideal line
plt.xlabel('Actual Electric Range')
plt.ylabel('Predicted Electric Range')
plt.title('Actual vs. Predicted Electric Range (Linear Regression)')
plt.show()

## 9. Model Evaluation



### 9.1 R-squared (R²) Score and MAE/RMSE



In [ ]:
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2) Score: {r2:.2f}")

### 9.2 Interpret R² Score



In [ ]:
if r2 < 0:
    print(f"The R-squared score is {r2:.2f}, which means the model is worse than simply predicting the mean of the target variable.")
elif r2 < 0.25:
    print(f"The R-squared score is {r2:.2f}, indicating a very weak fit. Less than 25% of the variance in electric range is explained by the model.")
elif r2 < 0.5:
    print(f"The R-squared score is {r2:.2f}, suggesting a weak fit. Less than 50% of the variance in electric range is explained by the model.")
elif r2 < 0.75:
    print(f"The R-squared score is {r2:.2f}, indicating a moderate fit. Approximately 51% of the variance in electric range is explained by the model.")
else:
    print(f"The R-squared score is {r2:.2f}, representing a strong fit. More than 75% of the variance in electric range is explained by the model.")

### 9.3 Analyze Regression Coefficients



In [ ]:
coefficients = pd.DataFrame({'feature': X.columns, 'coefficient': model.coef_})
coefficients['abs_coefficient'] = np.abs(coefficients['coefficient'])
coefficients = coefficients.sort_values(by='abs_coefficient', ascending=False)

print("--- Top 10 Features with Largest Absolute Coefficients (Strongest Impact) ---")
display(coefficients.head(10))

print("\n--- Top 10 Features with Largest Positive Coefficients (Increasing Electric Range) ---")
display(coefficients[coefficients['coefficient'] > 0].head(10))

print("\n--- Top 10 Features with Largest Negative Coefficients (Decreasing Electric Range) ---")
display(coefficients[coefficients['coefficient'] < 0].head(10))

### 9.5 Study Impact of Model Year



In [ ]:
model_year_coefficient = coefficients[coefficients['feature'] == 'model_year']['coefficient'].values

if len(model_year_coefficient) > 0:
    print(f"The coefficient for 'model_year' is: {model_year_coefficient[0]:.2f}")
    if model_year_coefficient[0] > 0:
        print("This suggests that, on average, newer model years are associated with a higher electric range.")
    else:
        print("This suggests that, on average, newer model years are associated with a lower electric range.")
else:
    print("The 'model_year' feature was not found in the model's coefficients.")